# ECOSORT QUEST AR — Kaggle GPU Training

Pipeline độc lập: audit/chia nhóm → head → fine-tune → chọn checkpoint/ngưỡng bằng validation → test → export cho backend.

**Trước Run All:** thêm dataset chứa `plastic/`, `metal/`, `paper/` bằng Add Input; Settings → Accelerator → GPU; bật Internet để tải trọng số ImageNet. Không cài lại TensorFlow/CUDA bằng pip. Notebook giữ phiên bản runtime thực tế trong output. Mặc định dùng một GPU (T4 x2 vẫn chỉ dùng GPU đầu), float32 để dễ chạy backend CPU.

Notebook chứa toàn bộ source, không cần upload code riêng. Chỉ dùng 3 lớp trên nếu input có thêm lớp. Không tự giải nén zip: upload bộ ảnh để Kaggle giải nén thành thư mục, hoặc giải nén an toàn trước. Không ghi vào `/kaggle/input`.

Đây là notebook chưa chạy trên Kaggle trong phiên làm việc hiện tại. Chạy trên GPU rồi tải bằng chứng thực tế; không coi số liệu baseline Windows là kết quả Kaggle.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.metadata
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["KERAS_HOME"] = "/kaggle/working/.keras"
os.environ["MPLCONFIGDIR"] = "/kaggle/working/.matplotlib"
import tensorflow as tf
print("Python:", sys.version)
print("TensorFlow:", tf.__version__, "Keras:", tf.keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "Enable GPU in Settings > Accelerator, then restart session."
assert int(tf.__version__.split(".")[0]) >= 2 and tuple(map(int, tf.__version__.split(".")[:2])) >= (2, 16), "Need modern TensorFlow/Keras .keras support."
for package in ("Pillow", "numpy", "scikit-learn", "matplotlib"):
    print(package, importlib.metadata.version(package))
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout or result.stderr)


## 1. Cấu hình — sửa trước khi chạy

Đặt `DATASET_ROOT` là thư mục **cha của ba thư mục lớp**, không phải đường dẫn zip. Để `None` nếu chỉ có đúng một bộ dữ liệu phù hợp trong Input. Chọn `RUN_NAME` mới cho mỗi lần train, không ghi đè model cũ. `GROUPS_CSV` tùy chọn có cột `path,group_id` cho vật phẩm/phiên chụp. Dataset hiện tại thiếu ID gốc nên dHash chỉ giảm rò rỉ, không chứng minh object-disjoint.


In [ ]:
DATASET_ROOT = None  # Ví dụ: Path("/kaggle/input/your-dataset/dataset-plastic-metal-paper")
GROUPS_CSV = None
RUN_NAME = "ecosort_gpu_v1"
CONFIG = {
    "seed": 42, "batch_size": 32,
    "head_epochs": 25, "fine_tune_epochs": 15,
    "head_lr": 1e-3, "fine_tune_lr": 1e-5, "unfreeze_last": 30,
    "threshold_minimum": 0.75, "target_accepted_accuracy": 0.90,
    "require_gpu": True, "weights": "imagenet",
}
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME, "Use a simple run folder name"
RUN = Path("/kaggle/working") / RUN_NAME
assert not RUN.exists(), "Run already exists. Choose a new RUN_NAME or restart a clean session."
if DATASET_ROOT is None:
    candidates = sorted({p.parent for p in Path("/kaggle/input").rglob("plastic")
                         if p.is_dir() and all((p.parent / c).is_dir() for c in ("plastic", "metal", "paper"))})
    print("Dataset candidates:", candidates)
    assert len(candidates) == 1, "Set DATASET_ROOT explicitly to one dataset root."
    DATASET_ROOT = candidates[0]
DATASET_ROOT = Path(DATASET_ROOT)
assert all((DATASET_ROOT / c).is_dir() for c in ("plastic", "metal", "paper"))
print("Input:", DATASET_ROOT, "Output:", RUN)


## 2. Source tự chứa — không cần sửa cell này

In [ ]:
CODE = Path("/kaggle/working/ecosort_code")
CODE.mkdir(exist_ok=True)
SOURCES = {'data.py': '"""Shared RGB/EXIF/resize rules for training, evaluation and inference."""\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nimport warnings\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps, UnidentifiedImageError\n\nCLASS_NAMES = ("Plastic", "Metal", "Paper")  # Serialized output order; never alphabetical.\nFOLDERS = ("plastic", "metal", "paper")\nIMAGE_SIZE = (224, 224)\nMAX_BYTES = 4 * 1024 * 1024\nMAX_PIXELS = 20_000_000\n\n\ndef decode_image(image_bytes: bytes, size: tuple[int, int] = IMAGE_SIZE) -> np.ndarray:\n    if not isinstance(image_bytes, bytes) or not image_bytes:\n        raise ValueError("image_bytes must be nonempty bytes")\n    if len(image_bytes) > MAX_BYTES:\n        raise ValueError("Image exceeds 4 MB upload limit")\n    try:\n        with warnings.catch_warnings():\n            warnings.simplefilter("error", Image.DecompressionBombWarning)\n            with Image.open(io.BytesIO(image_bytes)) as source:\n                if source.format not in ("JPEG", "PNG", "WEBP"):\n                    raise ValueError("Only JPEG, PNG and WEBP are supported")\n                if source.width * source.height > MAX_PIXELS:\n                    raise ValueError("Image exceeds pixel limit")\n                source.load()\n                rgb = ImageOps.exif_transpose(source).convert("RGB")\n                return np.asarray(rgb.resize(size, Image.Resampling.BILINEAR), dtype=np.uint8)\n    except (UnidentifiedImageError, OSError, Image.DecompressionBombError, Image.DecompressionBombWarning) as exc:\n        raise ValueError("Invalid or unsafe image") from exc\n\n\ndef image_fingerprints(path: Path) -> tuple[str, int, tuple[int, int]]:\n    with Image.open(path) as source:\n        source.load()\n        rgb = ImageOps.exif_transpose(source).convert("RGB")\n        digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()\n        gray = np.asarray(rgb.convert("L").resize((9, 8), Image.Resampling.LANCZOS))\n        bits = (gray[:, 1:] > gray[:, :-1]).flatten()\n        dhash = sum(int(value) << i for i, value in enumerate(bits))\n        return digest, dhash, rgb.size\n\n\ndef load_rows(rows: list[dict], dataset_root: Path) -> tuple[np.ndarray, np.ndarray]:\n    if not rows:\n        raise ValueError("Empty split")\n    images = np.stack([decode_image((dataset_root / row["path"]).read_bytes()) for row in rows])\n    labels = np.asarray([int(row["label"]) for row in rows], dtype=np.int64)\n    return images, labels\n', 'prepare_dataset.py': '"""Audit images and persist deterministic grouped 70/15/15 manifests, without copying images."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport random\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nfrom data import CLASS_NAMES, FOLDERS, image_fingerprints\n\n\ndef make_groups(records: list[dict], near_distance: int = 4, explicit: dict | None = None) -> list[list[dict]]:\n    parents = list(range(len(records)))\n\n    def find(i):\n        while parents[i] != i:\n            parents[i] = parents[parents[i]]\n            i = parents[i]\n        return i\n\n    def join(i, j):\n        parents[find(j)] = find(i)\n\n    for i, left in enumerate(records):\n        for j in range(i):\n            right = records[j]\n            same_pixels = left["sha256_pixels"] == right["sha256_pixels"]\n            if same_pixels and left["label"] != right["label"]:\n                raise ValueError(f"Conflicting labels for identical image: {left[\'path\']} / {right[\'path\']}")\n            same_group = explicit and explicit.get(left["path"]) is not None and explicit.get(left["path"]) == explicit.get(right["path"])\n            near = left["label"] == right["label"] and (left["dhash"] ^ right["dhash"]).bit_count() <= near_distance\n            if same_pixels or same_group or near:\n                join(i, j)\n    clusters = defaultdict(list)\n    for i, record in enumerate(records):\n        clusters[find(i)].append(record)\n    return list(clusters.values())\n\n\ndef assign_splits(groups: list[list[dict]], seed: int) -> list[dict]:\n    """Greedily balance class deficits, keeping whole (possibly cross-label) groups together."""\n    totals = Counter(int(row["label"]) for group in groups for row in group)\n    counts = {split: Counter() for split in ("train", "validation", "test")}\n    ratios = {"train": .70, "validation": .15, "test": .15}\n    ordered = list(groups)\n    random.Random(seed).shuffle(ordered)\n    ordered.sort(key=len, reverse=True)\n    rows = []\n    for index, group in enumerate(ordered):\n        composition = Counter(int(row["label"]) for row in group)\n        # Relative unfilled capacity; large groups placed first to limit fraction drift.\n        def deficit(split):\n            return sum(n * (ratios[split] * totals[label] - counts[split][label]) /\n                       max(1, ratios[split] * totals[label]) for label, n in composition.items())\n        split = max(ratios, key=deficit)\n        counts[split].update(composition)\n        for record in group:\n            rows.append({**record, "group_id": f"group-{index:05d}", "split": split})\n    for split in ratios:\n        if any(counts[split][label] == 0 for label in totals):\n            raise ValueError("Grouping leaves a class absent from a split. Review groups or collect more objects.")\n    return sorted(rows, key=lambda row: row["path"])\n\n\ndef prepare(root: Path, output: Path, seed=42, near_distance=4, groups_csv: Path | None = None):\n    records, invalid = [], []\n    for label, folder in enumerate(FOLDERS):\n        if not (root / folder).is_dir():\n            raise ValueError(f"Missing class directory: {root / folder}")\n        for path in sorted((root / folder).iterdir()):\n            if not path.is_file() or path.suffix.lower() not in (".jpg", ".jpeg", ".png", ".webp"):\n                continue\n            try:\n                digest, dhash, dimensions = image_fingerprints(path)\n                records.append({"path": path.relative_to(root).as_posix(), "label": label,\n                                "class_name": CLASS_NAMES[label], "sha256_pixels": digest, "dhash": dhash,\n                                "width": dimensions[0], "height": dimensions[1]})\n            except Exception as exc:\n                invalid.append({"path": str(path), "error": str(exc)})\n    if invalid:\n        raise ValueError(f"Invalid images must be reviewed before splitting: {invalid}")\n    explicit = None\n    if groups_csv:\n        with groups_csv.open(encoding="utf-8-sig", newline="") as stream:\n            explicit = {row["path"]: row["group_id"] for row in csv.DictReader(stream)}\n        if set(explicit) != {row["path"] for row in records}:\n            raise ValueError("groups CSV must cover all image paths exactly")\n    groups = make_groups(records, near_distance, explicit)\n    rows = assign_splits(groups, seed)\n    output.mkdir(parents=True, exist_ok=True)\n    with (output / "splits.csv").open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader(); writer.writerows(rows)\n    duplicate_groups = [group for group in groups if len(group) > 1]\n    audit = {"dataset_root": str(root.resolve()), "seed": seed, "class_names": CLASS_NAMES,\n             "total": len(records), "class_counts": dict(Counter(row["class_name"] for row in records)),\n             "split_counts": {split: dict(Counter(row["class_name"] for row in rows if row["split"] == split))\n                              for split in ("train", "validation", "test")},\n             "group_count": len(groups), "largest_group": max(map(len, groups)),\n             "exact_duplicate_excess": len(records) - len({row["sha256_pixels"] for row in records}),\n             "near_hash_distance": near_distance, "multi_image_groups": [[r["path"] for r in g] for g in duplicate_groups],\n             "explicit_object_session_groups": bool(explicit), "invalid_images": invalid,\n             "limitations": "No object/session metadata supplied. dHash grouping is heuristic, not proof of object-disjoint splits."\n             if not explicit else "Object/session grouping supplied by user; verify its correctness."}\n    (output / "dataset_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(json.dumps({k: v for k, v in audit.items() if k != "multi_image_groups"}, ensure_ascii=False, indent=2))\n    return audit\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--dataset", type=Path, default=Path(__file__).parent.parent / "Dataset/dataset-plastic-metal-paper")\n    parser.add_argument("--output", type=Path, default=Path(__file__).parent / "reports")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--near-distance", type=int, choices=range(0, 9), default=4)\n    parser.add_argument("--groups-csv", type=Path)\n    args = parser.parse_args()\n    prepare(args.dataset, args.output, args.seed, args.near_distance, args.groups_csv)\n', 'train.py': '"""MobileNetV2 transfer learning; validation-only model/threshold selection."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport platform\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\nos.environ.setdefault("TF_NUM_INTRAOP_THREADS", "4")\nos.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")\nos.environ.setdefault("KERAS_HOME", str(Path(__file__).parent / ".keras"))\nos.environ.setdefault("MPLCONFIGDIR", str(Path(__file__).parent / ".matplotlib"))\n\nimport numpy as np\nimport tensorflow as tf\nfrom sklearn.metrics import accuracy_score\n\nfrom data import CLASS_NAMES, IMAGE_SIZE, load_rows\n\n\ndef calibrate_threshold(probabilities, labels, minimum=.75, target_precision=.90):\n    confidence = probabilities.max(axis=1)\n    correct = probabilities.argmax(axis=1) == labels\n    candidates = []\n    for threshold in np.arange(minimum, .991, .01):\n        selected = confidence >= threshold\n        count = int(selected.sum())\n        precision = float(correct[selected].mean()) if count else None\n        candidates.append({"threshold": round(float(threshold), 2), "accepted": count,\n                           "coverage": count / len(labels), "accepted_accuracy": precision})\n    valid = [row for row in candidates if row["accepted"] >= 20 and row["accepted_accuracy"] >= target_precision]\n    chosen = max(valid, key=lambda row: (row["coverage"], -row["threshold"])) if valid else candidates[0]\n    return {"threshold": chosen["threshold"], "selection_split": "validation", "target_accepted_accuracy": target_precision,\n            "target_met": bool(valid), "chosen": chosen, "candidates": candidates,\n            "warning": "Confidence gating is not an out-of-distribution detector; only known-class validation images were available."}\n\n\ndef train(dataset: Path, manifest: Path, artifacts: Path, reports: Path, epochs=35, augmented_views=2,\n          fine_tune_epochs=0, seed=42, batch_size=32):\n    tf.keras.utils.set_random_seed(seed)\n    artifacts.mkdir(parents=True, exist_ok=True); reports.mkdir(parents=True, exist_ok=True)\n    if (artifacts / "model.keras").exists():\n        raise FileExistsError("Model already exists. Choose a new --artifacts directory to preserve the previous run.")\n    with manifest.open(encoding="utf-8", newline="") as stream:\n        rows = list(csv.DictReader(stream))\n    split_rows = {split: [row for row in rows if row["split"] == split] for split in ("train", "validation", "test")}\n    groups = {split: {row["group_id"] for row in data} for split, data in split_rows.items()}\n    if groups["train"] & groups["validation"] or groups["train"] & groups["test"] or groups["validation"] & groups["test"]:\n        raise ValueError("Group leakage in split manifest")\n    # Do not load or inspect test images during training/model selection.\n    x_train, y_train = load_rows(split_rows["train"], dataset)\n    x_val, y_val = load_rows(split_rows["validation"], dataset)\n    start = time.perf_counter()\n    backbone = tf.keras.applications.MobileNetV2(input_shape=(*IMAGE_SIZE, 3), include_top=False, weights="imagenet")\n    backbone.trainable = False\n    raw = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="rgb_0_255")\n    scaled = tf.keras.layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_preprocessing")(raw)\n    features = backbone(scaled, training=False)\n    pooled = tf.keras.layers.GlobalAveragePooling2D()(features)\n    encoder = tf.keras.Model(raw, pooled, name="imagenet_encoder")\n    feature_input = tf.keras.Input(shape=(1280,))\n    hidden = tf.keras.layers.Dense(128, activation="relu")(feature_input)\n    hidden = tf.keras.layers.Dropout(.35)(hidden)\n    output = tf.keras.layers.Dense(3, activation="softmax", name="class_probabilities")(hidden)\n    head = tf.keras.Model(feature_input, output, name="classification_head")\n    head.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])\n    augmentation = tf.keras.Sequential([\n        tf.keras.layers.RandomFlip("horizontal", seed=seed),\n        tf.keras.layers.RandomRotation(.06, fill_mode="reflect", seed=seed + 1),\n        tf.keras.layers.RandomZoom(.12, seed=seed + 2),\n        tf.keras.layers.RandomContrast(.12, seed=seed + 3),\n    ], name="train_only_augmentation")\n\n    def encode(images, augment=False):\n        batches = []\n        for offset in range(0, len(images), batch_size):\n            batch = tf.cast(images[offset:offset + batch_size], tf.float32)\n            if augment:\n                batch = tf.clip_by_value(augmentation(batch, training=True), 0, 255)\n            batches.append(encoder(batch, training=False).numpy())\n            if offset % (batch_size * 8) == 0:\n                print(f"Extract features: {min(offset + batch_size, len(images))}/{len(images)} augment={augment}", flush=True)\n        return np.concatenate(batches)\n\n    # Cache embeddings in RAM, not a train/validation/test mixing cache on disk.\n    train_features = [encode(x_train)]\n    for _ in range(augmented_views):\n        train_features.append(encode(x_train, augment=True))\n    train_features = np.concatenate(train_features)\n    train_labels = np.tile(y_train, augmented_views + 1)\n    val_features = encode(x_val)\n    counts = np.bincount(y_train, minlength=3)\n    class_weights = {i: float(len(y_train) / (3 * count)) for i, count in enumerate(counts)}\n    train_ds = tf.data.Dataset.from_tensor_slices((train_features, train_labels)).shuffle(len(train_labels), seed=seed).batch(batch_size)\n    val_ds = tf.data.Dataset.from_tensor_slices((val_features, y_val)).batch(batch_size)\n    history = head.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=class_weights,\n        callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),\n                   tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=.5)], verbose=2)\n    model = tf.keras.Model(raw, head(encoder(raw)), name="ecosort_mobilenetv2")\n    fine_history = None\n    baseline_val_loss = float(head.evaluate(val_ds, verbose=0)[0])\n    if fine_tune_epochs > 0:\n        baseline_weights = model.get_weights()\n        encoder.trainable = True; backbone.trainable = True\n        for layer in backbone.layers[:-30]:\n            layer.trainable = False\n        for layer in backbone.layers:\n            if isinstance(layer, tf.keras.layers.BatchNormalization):\n                layer.trainable = False\n        model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])\n        fine_train = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(len(y_train), seed=seed).batch(batch_size).map(\n            lambda x, y: (tf.clip_by_value(augmentation(tf.cast(x, tf.float32), training=True), 0, 255), y))\n        fine_val = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(batch_size)\n        fine_history = model.fit(fine_train, validation_data=fine_val, epochs=fine_tune_epochs, class_weight=class_weights,\n            callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)], verbose=2).history\n        if float(model.evaluate(fine_val, verbose=0)[0]) >= baseline_val_loss:\n            model.set_weights(baseline_weights)\n            print("Fine tuning did not improve validation loss; restored baseline weights.", flush=True)\n    val_probabilities = np.concatenate([model(x_val[i:i + batch_size], training=False).numpy()\n                                        for i in range(0, len(x_val), batch_size)])\n    calibration = calibrate_threshold(val_probabilities, y_val)\n    (reports / "threshold_calibration.json").write_text(json.dumps(calibration, indent=2), encoding="utf-8")\n    model.save(artifacts / "model.keras")\n    metadata = {"class_names": CLASS_NAMES, "input_shape": [224, 224, 3], "input_dtype": "float32",\n                "input_range": [0, 255], "preprocessing_in_model": "Rescaling(1/127.5, offset=-1)",\n                "resize": "Pillow bilinear, direct resize, EXIF transpose, RGB", "threshold": calibration["threshold"],\n                "threshold_target_met_on_validation": calibration["target_met"], "seed": seed,\n                "split_manifest_sha256": hashlib.sha256(manifest.read_bytes()).hexdigest(),\n                "model_sha256": hashlib.sha256((artifacts / "model.keras").read_bytes()).hexdigest(),\n                "tensorflow": tf.__version__, "keras": tf.keras.__version__, "python": platform.python_version(),\n                "epochs_requested": epochs, "epochs_run": len(history.history["loss"]),\n                "augmented_train_views": augmented_views, "fine_tune_epochs_requested": fine_tune_epochs,\n                "training_seconds": round(time.perf_counter() - start, 2),\n                "validation_accuracy": float(accuracy_score(y_val, val_probabilities.argmax(axis=1))),\n                "selection": "best validation loss; test set not loaded by train.py"}\n    (artifacts / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\n    (reports / "training_history.json").write_text(json.dumps({"head": history.history, "fine_tune": fine_history}, indent=2), encoding="utf-8")\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n    for ax, metric in zip(axes, ("loss", "accuracy")):\n        ax.plot(history.history[metric], label="Train (augmented features)")\n        ax.plot(history.history["val_" + metric], label="Validation"); ax.set_title(metric.title()); ax.set_xlabel("Epoch"); ax.legend()\n    fig.tight_layout(); fig.savefig(reports / "learning_curves.png", dpi=160); plt.close(fig)\n    print(json.dumps(metadata, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    base = Path(__file__).parent\n    parser.add_argument("--dataset", type=Path, default=base.parent / "Dataset/dataset-plastic-metal-paper")\n    parser.add_argument("--manifest", type=Path, default=base / "reports/splits.csv")\n    parser.add_argument("--artifacts", type=Path, default=base / "artifacts")\n    parser.add_argument("--reports", type=Path, default=base / "reports")\n    parser.add_argument("--epochs", type=int, default=35)\n    parser.add_argument("--augmented-views", type=int, default=2)\n    parser.add_argument("--fine-tune-epochs", type=int, default=0)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--batch-size", type=int, default=32)\n    args = parser.parse_args()\n    train(args.dataset, args.manifest, args.artifacts, args.reports, args.epochs, args.augmented_views,\n          args.fine_tune_epochs, args.seed, args.batch_size)\n', 'gpu_train.py': '"""GPU transfer learning for Kaggle; outputs compatible with existing inference.py."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport os\nimport platform\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\nimport numpy as np\nimport tensorflow as tf\nfrom data import CLASS_NAMES, load_rows, image_fingerprints\nfrom train import calibrate_threshold\nfrom evaluation_curves import diagnostic_curves, training_curves, split_distribution\n\n\nDEFAULT_CONFIG = {"seed": 42, "batch_size": 32, "head_epochs": 25, "fine_tune_epochs": 15,\n                  "head_lr": 1e-3, "fine_tune_lr": 1e-5, "unfreeze_last": 30,\n                  "require_gpu": True, "weights": "imagenet", "threshold_minimum": .75,\n                  "target_accepted_accuracy": .90}\n\n\nclass LearningRateLog(tf.keras.callbacks.Callback):\n    def on_epoch_end(self, epoch, logs=None):\n        if logs is not None:\n            logs["learning_rate"] = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))\n\n\ndef train_gpu(dataset, run, config=None):\n    dataset, run = Path(dataset), Path(run)\n    cfg = {**DEFAULT_CONFIG, **(config or {})}\n    if cfg["head_epochs"] < 1 or cfg["fine_tune_epochs"] < 0 or cfg["batch_size"] < 1 or cfg["unfreeze_last"] < 1:\n        raise ValueError("Invalid epoch/batch/unfreeze configuration")\n    gpus = tf.config.list_physical_devices("GPU")\n    if cfg["require_gpu"] and not gpus:\n        raise RuntimeError("No TensorFlow GPU detected. Enable Kaggle Settings > Accelerator > GPU and restart session.")\n    for gpu in gpus:\n        try:\n            tf.config.experimental.set_memory_growth(gpu, True)\n        except RuntimeError:\n            pass  # Notebook may have initialized devices already.\n    # Float32 keeps the exported model suitable for the CPU backend too.\n    # Kaggle automatically places supported operations on the first GPU.\n    tf.keras.mixed_precision.set_global_policy("float32")\n    tf.keras.utils.set_random_seed(cfg["seed"])\n    reports, artifacts = run / "reports", run / "artifacts"\n    manifest = reports / "splits.csv"\n    if (artifacts / "model.keras").exists() or (run / "checkpoints").exists():\n        raise FileExistsError("Run already trained/started; choose a NEW run directory to preserve evidence")\n    with manifest.open(encoding="utf-8", newline="") as stream:\n        rows = list(csv.DictReader(stream))\n    sets = {name: [r for r in rows if r["split"] == name] for name in ("train", "validation", "test")}\n    if any({int(r["label"]) for r in part} != {0, 1, 2} for part in sets.values()):\n        raise ValueError("Every split must contain all three classes")\n    if len({r["path"] for r in rows}) != len(rows):\n        raise ValueError("Repeated path in manifest")\n    for key in ("group_id", "sha256_pixels"):\n        seen = {}\n        for r in rows:\n            if seen.setdefault(r[key], r["split"]) != r["split"]:\n                raise ValueError(f"Split leakage: {key}")\n    # Audit hashes, not test predictions; prevents silently training with changed files.\n    for r in rows:\n        if image_fingerprints(dataset / r["path"])[0] != r["sha256_pixels"]:\n            raise ValueError(f"Image changed since split preparation: {r[\'path\']}")\n    artifacts.mkdir(parents=True); (run / "checkpoints").mkdir()\n    (run / "config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")\n    (run / "requirements-runtime.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")\n    (reports / "environment.json").write_text(json.dumps({"python": platform.python_version(), "tensorflow": tf.__version__,\n        "keras": tf.keras.__version__, "gpus": [g.name for g in gpus], "platform": platform.platform(),\n        "tensorflow_build": tf.sysconfig.get_build_info()}, indent=2, default=str), encoding="utf-8")\n    split_distribution(rows, reports)\n    x_train, y_train = load_rows(sets["train"], dataset)\n    x_val, y_val = load_rows(sets["validation"], dataset)\n    augmentation = tf.keras.Sequential([\n        tf.keras.layers.RandomFlip("horizontal", seed=cfg["seed"]),\n        tf.keras.layers.RandomRotation(.06, seed=cfg["seed"] + 1),\n        tf.keras.layers.RandomZoom(.12, seed=cfg["seed"] + 2),\n        tf.keras.layers.RandomContrast(.12, seed=cfg["seed"] + 3)], name="train_only_augmentation")\n    train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(len(y_train), seed=cfg["seed"])\n    train_ds = train_ds.batch(cfg["batch_size"]).map(\n        lambda x, y: (tf.clip_by_value(augmentation(tf.cast(x, tf.float32), training=True), 0, 255), y),\n        num_parallel_calls=1).prefetch(tf.data.AUTOTUNE)\n    val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(cfg["batch_size"]).prefetch(tf.data.AUTOTUNE)\n    backbone = tf.keras.applications.MobileNetV2(include_top=False, weights=cfg["weights"], input_shape=(224, 224, 3))\n    backbone.trainable = False\n    inputs = tf.keras.Input((224, 224, 3), name="rgb_0_255")\n    x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_preprocessing")(inputs)\n    x = backbone(x, training=False)  # BN statistics remain fixed in BOTH stages.\n    x = tf.keras.layers.GlobalAveragePooling2D()(x)\n    x = tf.keras.layers.Dense(128, activation="relu")(x)\n    x = tf.keras.layers.Dropout(.35)(x)\n    outputs = tf.keras.layers.Dense(3, activation="softmax", dtype="float32", name="class_probabilities")(x)\n    model = tf.keras.Model(inputs, outputs, name="ecosort_mobilenetv2")\n    counts = np.bincount(y_train, minlength=3)\n    class_weights = {i: float(len(y_train) / (3 * counts[i])) for i in range(3)}\n    histories = {}; start = time.perf_counter()\n\n    def fit_phase(phase, epochs, lr, patience):\n        model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])\n        path = run / "checkpoints" / f"{phase}.keras"\n        history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, class_weight=class_weights, shuffle=False,\n            callbacks=[tf.keras.callbacks.ModelCheckpoint(path, monitor="val_loss", save_best_only=True),\n                       tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),\n                       tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=.5, min_lr=lr / 100),\n                       LearningRateLog(), tf.keras.callbacks.CSVLogger(run / "reports" / f"{phase}_epochs.csv")], verbose=2)\n        histories[phase] = {k: [float(v) for v in values] for k, values in history.history.items()}\n        return float(model.evaluate(val_ds, verbose=0)[0])\n\n    head_loss = fit_phase("head", cfg["head_epochs"], cfg["head_lr"], 5)\n    selected = "head"; fine_loss = None\n    if cfg["fine_tune_epochs"]:\n        backbone.trainable = True\n        for layer in backbone.layers:\n            layer.trainable = False\n        for layer in backbone.layers[-cfg["unfreeze_last"]:]:\n            if not isinstance(layer, tf.keras.layers.BatchNormalization):\n                layer.trainable = True\n        fine_loss = fit_phase("fine_tune", cfg["fine_tune_epochs"], cfg["fine_tune_lr"], 4)\n        if fine_loss < head_loss:\n            selected = "fine_tune"\n    # Explicitly reload selected checkpoint; failed fine-tuning never replaces the head baseline.\n    model = tf.keras.models.load_model(run / "checkpoints" / f"{selected}.keras", compile=False, safe_mode=True)\n    val_probabilities = model.predict(val_ds, verbose=0)\n    calibration = calibrate_threshold(val_probabilities, y_val, cfg["threshold_minimum"], cfg["target_accepted_accuracy"])\n    (reports / "threshold_calibration.json").write_text(json.dumps(calibration, indent=2), encoding="utf-8")\n    diagnostic_curves(y_val, val_probabilities, reports / "validation", calibration["threshold"], "validation")\n    (reports / "training_history.json").write_text(json.dumps(histories, indent=2), encoding="utf-8")\n    training_curves(histories, reports)\n    model.save(artifacts / "model.keras")\n    metadata = {"class_names": CLASS_NAMES, "input_shape": [224, 224, 3], "input_dtype": "float32", "input_range": [0, 255],\n        "preprocessing_in_model": "Rescaling(1/127.5, offset=-1)", "resize": "Pillow bilinear, direct resize, EXIF transpose, RGB",\n        "threshold": calibration["threshold"], "threshold_target_met_on_validation": calibration["target_met"],\n        "model_sha256": hashlib.sha256((artifacts / "model.keras").read_bytes()).hexdigest(),\n        "split_manifest_sha256": hashlib.sha256(manifest.read_bytes()).hexdigest(), "seed": cfg["seed"],\n        "tensorflow": tf.__version__, "keras": tf.keras.__version__, "python": platform.python_version(),\n        "selected_stage": selected, "head_validation_loss": head_loss, "fine_tune_validation_loss": fine_loss,\n        "validation_accuracy": float((val_probabilities.argmax(1) == y_val).mean()),\n        "epochs_run": {k: len(v["loss"]) for k, v in histories.items()}, "training_seconds": time.perf_counter() - start,\n        "selection": "Lowest validation loss between stage checkpoints; train_gpu does not load test tensors",\n        "gpu_devices": [g.name for g in gpus], "precision_policy": "float32"}\n    (artifacts / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\n    # Reload the EXACT exported artifact through the backend interface before test evaluation.\n    from inference import Predictor\n    serving = Predictor(artifacts)\n    sample = (dataset / sets["validation"][0]["path"]).read_bytes()\n    response = serving.predict(sample)\n    (reports / "inference_smoke.json").write_text(json.dumps(response, indent=2), encoding="utf-8")\n    print(json.dumps(metadata, indent=2)); return metadata\n', 'evaluation_curves.py': '"""Diagnostic curves for a fixed multiclass classifier; never select thresholds on test."""\nfrom pathlib import Path\nimport json\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve,\n                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay)\nfrom data import CLASS_NAMES\n\n\ndef diagnostic_curves(labels, probabilities, output, threshold, split="test"):\n    labels, probabilities = np.asarray(labels), np.asarray(probabilities)\n    if probabilities.shape != (len(labels), 3) or set(labels.tolist()) != {0, 1, 2}:\n        raise ValueError("Curves require all three labeled classes and Nx3 probabilities")\n    if not np.isfinite(probabilities).all() or np.any(probabilities < 0) or not np.allclose(probabilities.sum(1), 1, atol=1e-4):\n        raise ValueError("Invalid probabilities")\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    truth = np.eye(3)[labels]\n    predicted = probabilities.argmax(1)\n    confidence = probabilities.max(1)\n    correct = predicted == labels\n    summary = {"split": split, "threshold_from_validation": float(threshold), "classes": {}}\n\n    def save(fig, name):\n        fig.tight_layout(); fig.savefig(output / name, dpi=170); plt.close(fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 5))\n    for i, name in enumerate(CLASS_NAMES):\n        fpr, tpr, _ = roc_curve(truth[:, i], probabilities[:, i])\n        precision, recall, _ = precision_recall_curve(truth[:, i], probabilities[:, i])\n        auc = float(roc_auc_score(truth[:, i], probabilities[:, i]))\n        ap = float(average_precision_score(truth[:, i], probabilities[:, i]))\n        summary["classes"][name] = {"roc_auc_ovr": auc, "average_precision": ap,\n                                   "fpr": fpr.tolist(), "tpr": tpr.tolist(),\n                                   "precision": precision.tolist(), "recall": recall.tolist()}\n        axes[0].plot(fpr, tpr, label=f"{name} AUC={auc:.3f}")\n        axes[1].plot(recall, precision, label=f"{name} AP={ap:.3f}")\n        axes[1].axhline(truth[:, i].mean(), linestyle=":", alpha=.25)\n    summary["macro_roc_auc_ovr"] = float(np.mean([v["roc_auc_ovr"] for v in summary["classes"].values()]))\n    summary["macro_average_precision"] = float(np.mean([v["average_precision"] for v in summary["classes"].values()]))\n    axes[0].plot([0, 1], [0, 1], "k--", alpha=.5)\n    axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title=f"One-vs-rest ROC ({split})")\n    axes[1].set(xlabel="Recall", ylabel="Precision", title=f"One-vs-rest Precision-Recall ({split})")\n    for ax in axes:\n        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.legend(); ax.grid(alpha=.2)\n    save(fig, "roc_pr_curves.png")\n\n    fig, ax = plt.subplots(figsize=(6, 5))\n    ConfusionMatrixDisplay(confusion_matrix(labels, predicted, labels=[0, 1, 2], normalize="true"),\n                           display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)\n    ax.set_title(f"Confusion matrix normalized by true class ({split})")\n    save(fig, "confusion_matrix_normalized.png")\n\n    bins = []\n    for lower, upper in zip(np.linspace(0, 1, 11)[:-1], np.linspace(0, 1, 11)[1:]):\n        mask = (confidence >= lower) & ((confidence < upper) if upper < 1 else (confidence <= upper))\n        if mask.any():\n            bins.append({"lower": float(lower), "upper": float(upper), "count": int(mask.sum()),\n                         "confidence": float(confidence[mask].mean()), "accuracy": float(correct[mask].mean())})\n    summary["top_label_calibration_bins"] = bins\n    summary["ece_10_equal_width_bins"] = float(sum(b["count"] / len(labels) * abs(b["confidence"] - b["accuracy"]) for b in bins))\n    summary["multiclass_brier_score_sum_over_classes"] = float(np.mean(np.sum((probabilities - truth) ** 2, axis=1)))\n    fig, axes = plt.subplots(1, 2, figsize=(12, 5))\n    axes[0].plot([0, 1], [0, 1], "k--", label="Ideal")\n    axes[0].plot([b["confidence"] for b in bins], [b["accuracy"] for b in bins], "o-", label="Top-label bins")\n    axes[0].set(xlabel="Mean confidence", ylabel="Observed accuracy", xlim=(0, 1), ylim=(0, 1.02),\n                title=f"Reliability ({split}), ECE={summary[\'ece_10_equal_width_bins\']:.3f}")\n    axes[0].legend()\n    for mask, name in ((correct, "Correct"), (~correct, "Incorrect")):\n        axes[1].hist(confidence[mask], bins=np.linspace(0, 1, 21), alpha=.6, label=name)\n    axes[1].axvline(threshold, color="black", linestyle="--", label=f"Validation threshold={threshold:.2f}")\n    axes[1].set(xlabel="Maximum softmax probability", ylabel="Images", title="Confidence distribution")\n    axes[1].legend()\n    save(fig, "reliability_confidence.png")\n\n    points = []\n    for value in sorted(set(np.linspace(0, 1, 101).tolist() + [float(threshold)])):\n        mask = confidence >= value\n        points.append({"threshold": value, "accepted": int(mask.sum()), "coverage": float(mask.mean()),\n                       "accepted_accuracy": float(correct[mask].mean()) if mask.any() else None})\n    summary["threshold_diagnostics_not_for_test_selection"] = points\n    fig, axes = plt.subplots(1, 2, figsize=(12, 5))\n    axes[0].plot([p["threshold"] for p in points], [p["coverage"] for p in points], label="Coverage")\n    axes[0].plot([p["threshold"] for p in points], [np.nan if p["accepted_accuracy"] is None else p["accepted_accuracy"] for p in points], label="Accepted accuracy")\n    axes[0].axvline(threshold, linestyle="--", color="black", label="Chosen on validation")\n    axes[0].set(xlabel="Confidence threshold", ylabel="Fraction", ylim=(0, 1.02), title=f"Threshold diagnostics ({split})")\n    axes[0].legend()\n    valid = sorted([p for p in points if p["accepted_accuracy"] is not None], key=lambda p: p["coverage"])\n    axes[1].plot([p["coverage"] for p in valid], [1 - p["accepted_accuracy"] for p in valid], "o-", markersize=2)\n    axes[1].set(xlabel="Coverage", ylabel="Selective risk (1 - accepted accuracy)", title=f"Risk-coverage ({split})", xlim=(0, 1), ylim=(0, 1))\n    save(fig, "threshold_risk_coverage.png")\n    summary["limitations"] = "Known-class diagnostic curves only. Confidence/ECE do not establish OOD rejection; no test-based threshold selection. Small bins may be noisy."\n    (output / "curve_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n    np.savez_compressed(output / "probabilities.npz", labels=labels, probabilities=probabilities)\n    return summary\n\n\ndef training_curves(histories, output):\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    fig, axes = plt.subplots(1, 3, figsize=(16, 4))\n    offset = 0\n    for phase, history in histories.items():\n        count = len(history["loss"]); epochs = np.arange(offset + 1, offset + count + 1)\n        for ax, metric in zip(axes[:2], ("loss", "accuracy")):\n            ax.plot(epochs, history[metric], label=f"{phase}: train (augmentation)")\n            ax.plot(epochs, history["val_" + metric], label=f"{phase}: validation")\n            ax.set_title(metric.title()); ax.set_xlabel("Epoch"); ax.legend(fontsize=7)\n        axes[2].plot(epochs, history.get("learning_rate", [np.nan] * count), label=phase)\n        if offset:\n            for ax in axes:\n                ax.axvline(offset + .5, linestyle="--", color="grey")\n        offset += count\n    axes[2].set(title="Learning rate", xlabel="Epoch", yscale="log"); axes[2].legend()\n    for ax in axes:\n        ax.grid(alpha=.2)\n    fig.tight_layout(); fig.savefig(output / "training_curves.png", dpi=170); plt.close(fig)\n\n\ndef split_distribution(rows, output):\n    from collections import Counter\n    fig, ax = plt.subplots(figsize=(8, 4))\n    x = np.arange(3)\n    for i, split in enumerate(("train", "validation", "test")):\n        counts = Counter(int(row["label"]) for row in rows if row["split"] == split)\n        bars = ax.bar(x + (i - 1) * .25, [counts[j] for j in range(3)], .25, label=split)\n        ax.bar_label(bars, fontsize=8)\n    ax.set_xticks(x, CLASS_NAMES); ax.set(ylabel="Images", title="Dataset distribution by split"); ax.legend()\n    fig.tight_layout(); fig.savefig(Path(output) / "split_distribution.png", dpi=170); plt.close(fig)\n', 'inference.py': '"""Backend handoff: predict_image(image_bytes: bytes) -> dict; no FastAPI dependency."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport threading\nfrom pathlib import Path\n\nimport numpy as np\n\nif __package__:\n    from .data import CLASS_NAMES, decode_image\nelse:\n    from data import CLASS_NAMES, decode_image\n\n\nclass Predictor:\n    def __init__(self, artifacts: Path | str | None = None, threshold: float | None = None):\n        root = Path(artifacts or os.environ.get("ECOSORT_AI_ARTIFACTS", Path(__file__).parent / "artifacts"))\n        model_path, metadata_path = root / "model.keras", root / "metadata.json"\n        if not model_path.is_file() or not metadata_path.is_file():\n            raise RuntimeError("AI artifacts missing: need model.keras and metadata.json")\n        self.metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n        self.classes = tuple(self.metadata["class_names"])\n        if self.classes != CLASS_NAMES or self.metadata["input_shape"] != [224, 224, 3]:\n            raise RuntimeError("Unsupported model class mapping or input shape")\n        if hashlib.sha256(model_path.read_bytes()).hexdigest() != self.metadata["model_sha256"]:\n            raise RuntimeError("Model checksum does not match metadata")\n        self.threshold = float(threshold if threshold is not None else os.environ.get("ECOSORT_AI_THRESHOLD", self.metadata["threshold"]))\n        if not np.isfinite(self.threshold) or not 0 <= self.threshold <= 1:\n            raise ValueError("Threshold must be finite and between 0 and 1")\n        os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\n        import tensorflow as tf\n        self.model = tf.keras.models.load_model(model_path, compile=False, safe_mode=True)\n        if tuple(self.model.input_shape[1:]) != (224, 224, 3) or self.model.output_shape[-1] != 3:\n            raise RuntimeError("Model dimensions do not match metadata")\n        self.lock = threading.Lock()\n\n    def predict(self, image_bytes: bytes) -> dict:\n        image = decode_image(image_bytes).astype(np.float32)[None, ...]\n        # Rescaling is inside the exported model: do not call preprocess_input again.\n        with self.lock:\n            probabilities = np.asarray(self.model(image, training=False))[0]\n        if probabilities.shape != (3,) or not np.all(np.isfinite(probabilities)) or np.any(probabilities < 0) or not np.isclose(probabilities.sum(), 1, atol=1e-4):\n            raise RuntimeError("Model returned invalid probabilities")\n        index = int(probabilities.argmax())\n        confidence = float(probabilities[index])\n        return {"class_name": self.classes[index], "confidence": confidence, "accepted": bool(confidence >= self.threshold)}\n\n\n_predictor = None\n_load_lock = threading.Lock()\n\n\ndef predict_image(image_bytes: bytes) -> dict:\n    # Validate before loading a potentially large model, also on a cold backend request.\n    decode_image(image_bytes)\n    global _predictor\n    if _predictor is None:\n        with _load_lock:\n            if _predictor is None:\n                _predictor = Predictor()\n    return _predictor.predict(image_bytes)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("image", type=Path)\n    parser.add_argument("--artifacts", type=Path)\n    parser.add_argument("--threshold", type=float)\n    args = parser.parse_args()\n    try:\n        print(json.dumps(Predictor(args.artifacts, args.threshold).predict(args.image.read_bytes()), ensure_ascii=False))\n    except (ValueError, RuntimeError, OSError) as error:\n        parser.exit(1, str(error) + "\\n")\n', 'evaluate.py': '"""Evaluate a frozen model on the held-out test manifest or independent labeled phone photos."""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport platform\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\nos.environ.setdefault("TF_NUM_INTRAOP_THREADS", "4")\nos.environ.setdefault("TF_NUM_INTEROP_THREADS", "2")\nos.environ.setdefault("MPLCONFIGDIR", str(Path(__file__).parent / ".matplotlib"))\n\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport tensorflow as tf\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score\n\nfrom data import CLASS_NAMES, FOLDERS, image_fingerprints, load_rows\nfrom inference import Predictor\n\n\ndef evaluate(dataset: Path, manifest: Path, artifacts: Path, reports: Path, external: Path | None = None):\n    predictor = Predictor(artifacts)\n    metadata = predictor.metadata\n    if hashlib.sha256(manifest.read_bytes()).hexdigest() != metadata["split_manifest_sha256"]:\n        raise ValueError("Manifest differs from the one used to train this model")\n    with manifest.open(encoding="utf-8", newline="") as stream:\n        all_rows = list(csv.DictReader(stream))\n    if external:\n        known = {row["sha256_pixels"] for row in all_rows}\n        rows = []\n        for label, folder in enumerate(FOLDERS):\n            for path in sorted((external / folder).glob("*")):\n                if not path.is_file() or path.suffix.lower() not in (".png", ".jpg", ".jpeg", ".webp"):\n                    continue\n                digest, _, _ = image_fingerprints(path)\n                if digest in known:\n                    raise ValueError(f"External photo duplicates original dataset: {path}")\n                rows.append({"path": path.relative_to(external).as_posix(), "label": label})\n        if not rows or set(int(row["label"]) for row in rows) != {0, 1, 2}:\n            raise ValueError("External evaluation requires labeled plastic/, metal/ and paper/ images")\n        dataset = external\n    else:\n        rows = [row for row in all_rows if row["split"] == "test"]\n    images, labels = load_rows(rows, dataset)\n    probabilities = np.concatenate([np.asarray(predictor.model(images[i:i + 32], training=False))\n                                    for i in range(0, len(images), 32)])\n    predicted = probabilities.argmax(axis=1)\n    confidence = probabilities.max(axis=1)\n    accepted = confidence >= predictor.threshold\n    report = classification_report(labels, predicted, labels=[0, 1, 2], target_names=CLASS_NAMES,\n                                   output_dict=True, zero_division=0)\n    matrix = confusion_matrix(labels, predicted, labels=[0, 1, 2])\n    latencies = []\n    predictor.predict((dataset / rows[0]["path"]).read_bytes())  # Warmup, not measured.\n    for row in rows[:30]:\n        content = (dataset / row["path"]).read_bytes()\n        start = time.perf_counter(); predictor.predict(content)\n        latencies.append((time.perf_counter() - start) * 1000)\n    accuracy = float(accuracy_score(labels, predicted)); macro_f1 = float(f1_score(labels, predicted, average="macro"))\n    metrics = {"evaluation_set": "external_photos" if external else "held_out_test", "images": len(rows),\n               "accuracy": accuracy, "macro_f1": macro_f1, "macro_precision": report["macro avg"]["precision"],\n               "macro_recall": report["macro avg"]["recall"], "class_names": CLASS_NAMES,\n               "confusion_matrix": matrix.tolist(), "threshold": predictor.threshold,\n               "accepted_count": int(accepted.sum()), "coverage": float(accepted.mean()),\n               "accepted_accuracy": float((predicted[accepted] == labels[accepted]).mean()) if accepted.any() else None,\n               "all_images_metrics_include_rejected": True, "target_accuracy_85_met": accuracy >= .85,\n               "target_macro_f1_080_met": macro_f1 >= .80,\n               "latency_warm_median_ms": float(np.median(latencies)), "latency_warm_p95_ms": float(np.percentile(latencies, 95)),\n               "latency_samples": len(latencies), "latency_includes": "decode/EXIF/RGB/resize/model; excludes disk IO and HTTP",\n               "platform": platform.platform(), "tensorflow": tf.__version__, "gpu_devices": [d.name for d in tf.config.list_physical_devices("GPU")],\n               "model_sha256": metadata["model_sha256"], "split_manifest_sha256": metadata["split_manifest_sha256"]}\n    reports.mkdir(parents=True, exist_ok=True)\n    from evaluation_curves import diagnostic_curves\n    curve_summary = diagnostic_curves(labels, probabilities, reports, predictor.threshold,\n                                      "external_photos" if external else "test")\n    metrics["macro_roc_auc_ovr"] = curve_summary["macro_roc_auc_ovr"]\n    metrics["macro_average_precision"] = curve_summary["macro_average_precision"]\n    metrics["ece_10_equal_width_bins"] = curve_summary["ece_10_equal_width_bins"]\n    (reports / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")\n    (reports / "classification_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    with (reports / "predictions.csv").open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=["path", "true_class", "predicted_class", "confidence", "accepted", "correct"])\n        writer.writeheader()\n        for row, truth, prediction, score, accept in zip(rows, labels, predicted, confidence, accepted):\n            writer.writerow({"path": row["path"], "true_class": CLASS_NAMES[truth], "predicted_class": CLASS_NAMES[prediction],\n                             "confidence": float(score), "accepted": bool(accept), "correct": bool(truth == prediction)})\n    fig, ax = plt.subplots(figsize=(6, 5))\n    ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Greens", colorbar=False)\n    ax.set_title(f"EcoSort MobileNetV2 / {metrics[\'evaluation_set\']} (n={len(rows)})")\n    fig.tight_layout(); fig.savefig(reports / "confusion_matrix.png", dpi=180); plt.close(fig)\n    errors = np.flatnonzero(labels != predicted)[:12]\n    if len(errors):\n        fig, axes = plt.subplots(int(np.ceil(len(errors) / 4)), 4, figsize=(12, 3 * int(np.ceil(len(errors) / 4))), squeeze=False)\n        for ax in axes.flat:\n            ax.axis("off")\n        for ax, i in zip(axes.flat, errors):\n            ax.imshow(images[i]); ax.set_title(f"True: {CLASS_NAMES[labels[i]]} / Pred: {CLASS_NAMES[predicted[i]]}\\n{confidence[i]:.1%}", fontsize=10)\n        fig.suptitle("First misclassified test images (not training examples)"); fig.tight_layout()\n        fig.savefig(reports / "misclassified_examples.png", dpi=140); plt.close(fig)\n    per_class = "\\n".join(f"| {name} | {report[name][\'precision\']:.3f} | {report[name][\'recall\']:.3f} | {report[name][\'f1-score\']:.3f} | {int(report[name][\'support\'])} |" for name in CLASS_NAMES)\n    content = f"""# Báo cáo đánh giá EcoSort MobileNetV2\n\nModel: MobileNetV2 pretrained ImageNet + Global Average Pooling + Dense 128 + Dropout + Softmax 3 lớp.\nTập đánh giá: **{metrics[\'evaluation_set\']}**, **{len(rows)} ảnh**. Train/validation/test được cố định trước huấn luyện; chọn checkpoint và ngưỡng bằng validation, không dùng test để chọn model.\n\n| Chỉ số trên toàn bộ ảnh | Kết quả |\n|---|---:|\n| Accuracy | {accuracy:.2%} |\n| Macro Precision | {metrics[\'macro_precision\']:.4f} |\n| Macro Recall | {metrics[\'macro_recall\']:.4f} |\n| Macro F1 | {macro_f1:.4f} |\n| Mục tiêu nội bộ Accuracy ≥85% | {\'Đạt\' if accuracy >= .85 else \'Chưa đạt\'} |\n| Mục tiêu nội bộ Macro F1 ≥0.80 | {\'Đạt\' if macro_f1 >= .80 else \'Chưa đạt\'} |\n\nCác mục tiêu này không phải điểm chấm hoặc bảo đảm kết quả ngoài thực tế.\n\n| Lớp | Precision | Recall | F1 | Số ảnh |\n|---|---:|---:|---:|---:|\n{per_class}\n\n![Confusion matrix](confusion_matrix.png)\n\nNgưỡng confidence chọn từ validation: **{predictor.threshold:.2f}**. Coverage trên tập đánh giá: **{metrics[\'coverage\']:.2%}**; accepted accuracy: **{metrics[\'accepted_accuracy\']}**. Accuracy/Macro F1 phía trên tính cả ảnh bị từ chối, không chỉ tính ảnh confidence cao. Chỉ số accepted accuracy là độ chính xác phân loại trong các ảnh được nhận, không phải calibration hoặc OOD accuracy.\n\nThời gian inference sau warmup, bao gồm giải mã/resize và model, không gồm HTTP/đọc file: median **{metrics[\'latency_warm_median_ms\']:.1f} ms**, p95 **{metrics[\'latency_warm_p95_ms\']:.1f} ms**, {len(latencies)} ảnh. TensorFlow {tf.__version__}; GPU thấy bởi TensorFlow: {metrics[\'gpu_devices\']}.\n\n## Giới hạn\n\n- Dataset hiện tại có nền và điều kiện chụp hẹp. Kết quả này chưa chứng minh chất lượng trên ảnh điện thoại người dùng.\n- Không có object/session ID gốc; nhóm ảnh gần nhau bằng dHash chỉ giảm rủi ro rò rỉ, không bảo đảm object-disjoint.\n- Chỉ nhận diện một vật phẩm chính thuộc Plastic/Metal/Paper; không object detection hoặc phân loại nhiều vật phẩm.\n- Confidence cao không bảo đảm ảnh thuộc ba lớp. Chưa có tập unknown/OOD để đánh giá khả năng từ chối vật phẩm lạ.\n- Chưa kiểm thử Unity → FastAPI → model thật hoặc ảnh chụp điện thoại độc lập.\n\nModel SHA256: `{metadata[\'model_sha256\']}`.\nManifest SHA256: `{metadata[\'split_manifest_sha256\']}`.\nChi tiết từng ảnh: `predictions.csv`; số liệu máy đọc: `metrics.json`, `classification_report.json`.\n\n## Đồ thị bổ sung\n\n- `roc_pr_curves.png`: ROC và Precision–Recall one-vs-rest, AUC/AP từng lớp.\n- `confusion_matrix_normalized.png`: tỉ lệ theo nhãn thật.\n- `reliability_confidence.png`: reliability diagram và phân bố confidence (ECE chẩn đoán, không phải bảo đảm calibration/OOD).\n- `threshold_risk_coverage.png`: coverage, accepted accuracy và selective risk theo ngưỡng; chỉ chẩn đoán test, không dùng chọn lại ngưỡng.\n- `curve_metrics.json`, `probabilities.npz`: dữ liệu tái tạo đồ thị.\n"""\n    (reports / "EVALUATION.md").write_text(content, encoding="utf-8")\n    print(json.dumps(metrics, indent=2), flush=True)\n    return metrics\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    base = Path(__file__).parent\n    parser.add_argument("--dataset", type=Path, default=base.parent / "Dataset/dataset-plastic-metal-paper")\n    parser.add_argument("--manifest", type=Path, default=base / "reports/splits.csv")\n    parser.add_argument("--artifacts", type=Path, default=base / "artifacts")\n    parser.add_argument("--reports", type=Path, default=base / "reports")\n    parser.add_argument("--external-dataset", type=Path)\n    args = parser.parse_args()\n    if args.external_dataset and args.reports == base / "reports":\n        parser.error("Use a separate --reports directory for external evaluation to preserve the held-out test report")\n    evaluate(args.dataset, args.manifest, args.artifacts, args.reports, args.external_dataset)\n', '__init__.py': '"""EcoSort AI backend package (copy this directory as ai_model)."""\nif __package__:\n    from .inference import predict_image\nelse:\n    from inference import predict_image\n\n__all__ = ["predict_image"]\n'}
for name, source in SOURCES.items():
    (CODE / name).write_text(source, encoding="utf-8")
sys.path.insert(0, str(CODE))
print("Source modules written:", list(SOURCES))


## 3. Audit và chia Train/Validation/Test

Giữ toàn bộ nhóm trùng/gần trùng trong một split, seed 42, mục tiêu 70/15/15. Dừng khi ảnh lỗi hoặc nhãn xung đột. Augmentation chỉ dùng train. Tránh đổi seed/split để tìm điểm test tốt hơn. Tập test cục bộ đã được xem ở lần baseline trước; chạy lại cùng split giúp so sánh, nhưng không biến nó thành một test độc lập mới.


In [ ]:
from prepare_dataset import prepare
audit = prepare(DATASET_ROOT, RUN / "reports", seed=CONFIG["seed"], near_distance=4,
                groups_csv=Path(GROUPS_CSV) if GROUPS_CSV else None)
print(json.dumps(audit["split_counts"], indent=2))


## 4. Train head và fine-tune GPU

EarlyStopping, ReduceLROnPlateau, best checkpoint mỗi giai đoạn; fine-tune 30 layer cuối, BatchNorm cố định. Chọn checkpoint có **validation loss thấp nhất** giữa hai giai đoạn. Nếu fine-tune kém hơn, tự khôi phục head. Input raw RGB 0–255, chuẩn hóa [-1,1] nằm trong model; export không chứa augmentation. Không tải tensor test trong bước này.

Nếu OOM: tạo run mới và giảm batch 32 → 16 → 8. Không cần bật mixed precision để dùng GPU. Không bảo đảm fine-tune sẽ tốt hơn baseline.


In [ ]:
from gpu_train import train_gpu
metadata = train_gpu(DATASET_ROOT, RUN, CONFIG)
print("Selected stage:", metadata["selected_stage"])
print("Validation accuracy:", metadata["validation_accuracy"])
print("Threshold:", metadata["threshold"])


## 5. Xem validation và đường học trước khi test

Loss/Accuracy cho cả hai phase, Learning Rate, phân bố dữ liệu; đồ thị validation nằm trong `reports/validation`. Train có augmentation/dropout, validation không có, nên hai đường không hoàn toàn cùng điều kiện đo. Nếu cần chỉnh cấu hình, chỉ dựa train/validation và chạy run mới **trước khi mở cell test**.


In [ ]:
from IPython.display import display, Image
for name in ("split_distribution.png", "training_curves.png"):
    display(Image(filename=str(RUN / "reports" / name)))
for name in ("roc_pr_curves.png", "reliability_confidence.png", "threshold_risk_coverage.png"):
    display(Image(filename=str(RUN / "reports/validation" / name)))
print((RUN / "reports/threshold_calibration.json").read_text())


## 6. Test giữ riêng — chỉ chạy sau khi chốt cấu hình

Tải lại model `.keras` qua chính `Predictor` dùng cho backend, xác minh checksum và manifest. Báo cáo Accuracy/Precision/Recall/F1 trên **mọi ảnh**, kể cả ảnh rejected. ROC/PR là one-vs-rest; PR có baseline theo tỉ lệ lớp. Reliability/ECE và confidence chỉ chẩn đoán, không chứng minh unknown/OOD rejection. Risk–coverage trên test không được dùng chọn lại ngưỡng.


In [ ]:
from evaluate import evaluate
metrics = evaluate(DATASET_ROOT, RUN / "reports/splits.csv", RUN / "artifacts", RUN / "reports")
print((RUN / "reports/EVALUATION.md").read_text())
for name in ("confusion_matrix.png", "confusion_matrix_normalized.png", "roc_pr_curves.png",
             "reliability_confidence.png", "threshold_risk_coverage.png", "misclassified_examples.png"):
    if (RUN / "reports" / name).exists():
        display(Image(filename=str(RUN / "reports" / name)))


## 7. Đóng gói và tải kết quả

Output gồm model + metadata cùng lần train, source inference, requirements runtime, cấu hình, manifest và các báo cáo. Không có dataset, checkpoint trung gian hoặc CUDA driver trong zip. `requirements-runtime.txt` là ghi nhận Kaggle, **không cài mù toàn bộ lên backend Windows**; cài TensorFlow/Keras tương thích theo phiên bản metadata rồi kiểm thử môi trường đích.

Sau cell này chọn Save Version → Save & Run All để lưu kết quả của lần chạy hoàn chỉnh, rồi tải zip ở Output. Bảo đảm RUN_NAME mới nếu chạy lại trong session hiện tại. Huấn luyện không đồng nghĩa kiểm thử điện thoại; cần bước trong hướng dẫn tích hợp.


In [ ]:
import shutil, hashlib, zipfile
EXPORT = RUN / "export" / "ai_model"
EXPORT.mkdir(parents=True)
for name in ("__init__.py", "inference.py", "data.py"):
    shutil.copy2(CODE / name, EXPORT / name)
shutil.copytree(RUN / "artifacts", EXPORT / "artifacts")
shutil.copytree(RUN / "reports", EXPORT / "reports")
shutil.copytree(CODE, EXPORT / "training_source", ignore=shutil.ignore_patterns("__pycache__"))
for name in ("config.json", "requirements-runtime.txt"):
    shutil.copy2(RUN / name, EXPORT / name)
(EXPORT / "requirements-serving.txt").write_text(
    f"tensorflow=={tf.__version__}\nkeras=={tf.keras.__version__}\nPillow>=11,<13\nnumpy>=1.26,<3\n", encoding="utf-8")
checksums = {p.relative_to(EXPORT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
             for p in EXPORT.rglob("*") if p.is_file()}
(EXPORT / "SHA256SUMS.json").write_text(json.dumps(checksums, indent=2))
archive = shutil.make_archive(str(RUN / "ecosort-kaggle-handoff"), "zip", RUN / "export")
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    for name, digest in checksums.items():
        assert hashlib.sha256(z.read("ai_model/" + name)).hexdigest() == digest
print("Download:", archive)
print("SHA256:", hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from IPython.display import FileLink
display(FileLink(archive))


## Đọc kết quả và bước tiếp theo

- Loss train giảm nhưng val tăng: dấu hiệu overfit; checkpoint sớm đã được giữ, không chọn epoch cuối chỉ vì train accuracy cao.
- ROC AUC/AP cao không thay thế accuracy/F1 tại quyết định argmax. Confusion matrix cho biết cặp lớp hay nhầm.
- ECE thấp trên dataset nhỏ không đảm bảo calibration trên điện thoại. Coverage giảm có thể tăng accepted accuracy nhưng cũng từ chối nhiều ảnh hơn.
- Chưa có khoảng tin cậy theo vật phẩm/session vì thiếu ID gốc; không khẳng định độ chính xác thực địa.
- Download zip → thành viên 3 nạp package `ai_model` trong FastAPI `/predict` → Unity đổi ApiSettings sang Rest. Xem `INTEGRATION-UNITY-AI.md` trong workspace.

Nguồn: [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks), [TensorFlow transfer learning](https://www.tensorflow.org/guide/keras/transfer_learning), [MobileNetV2](https://www.tensorflow.org/api_docs/python/tf/keras/applications/MobileNetV2).
